# Autoresearch Test Notebook
Verifies everything needed for autoresearch works.

**Local tests** (1-6): no GPU, run anywhere.
**Cluster tests** (7-9): need GPU, auto-skip locally.

```bash
# Local
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/test_autoresearch.ipynb

# Cluster (GPU)
srun --partition=dev_gpu_h100 --time=00:10:00 --gres=gpu:1 --mem=64G \
  uv run jupyter nbconvert --to notebook --execute --inplace notebooks/test_autoresearch.ipynb
```

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), 'src'))
print('Setup OK')

Setup OK


## Test 1: Tower of Hanoi — YAML prompt loading

In [2]:
from tower_of_hanoi.enviroment import TowerOfHanoi
from tower_of_hanoi import prompts as toh_prompts

game = TowerOfHanoi(num_disks=3)

for variant in ['base', 'cot_detailed', 'minimal']:
    sys_prompt = toh_prompts.get_system_prompt(game, variant=variant)
    usr_prompt = toh_prompts.build_user_prompt(
        current_state=game.get_state(),
        previous_move='None',
        environment=game,
        step=1,
        variant=variant,
    )
    assert '{num_disks}' not in sys_prompt, f'Unfilled placeholder in {variant} system prompt'
    assert '{current_state}' not in usr_prompt, f'Unfilled placeholder in {variant} user prompt'
    print(f'  {variant}: OK (sys={len(sys_prompt)} chars, usr={len(usr_prompt)} chars)')

print('PASS: All ToH prompt variants load and format correctly')

  base: OK (sys=1151 chars, usr=822 chars)
  cot_detailed: OK (sys=745 chars, usr=516 chars)
  minimal: OK (sys=214 chars, usr=265 chars)
PASS: All ToH prompt variants load and format correctly


## Test 2: Sliding Puzzle — YAML prompt loading

In [3]:
from sliding_puzzle.enviroment import SlidingPuzzle, CONFIGS
from sliding_puzzle import prompts as sp_prompts

game = SlidingPuzzle(initial_state=CONFIGS['3x3 (easiest)'])

for variant in ['base', 'cot_detailed', 'minimal']:
    sys_prompt = sp_prompts.get_system_prompt(game, variant=variant)
    usr_prompt = sp_prompts.build_user_prompt(
        current_state=game.get_state(),
        previous_move='None',
        environment=game,
        step=1,
        variant=variant,
    )
    assert '{n}' not in sys_prompt, f'Unfilled placeholder in {variant} system prompt'
    assert '{current_state}' not in usr_prompt, f'Unfilled placeholder in {variant} user prompt'
    print(f'  {variant}: OK (sys={len(sys_prompt)} chars, usr={len(usr_prompt)} chars)')

print('PASS: All SP prompt variants load and format correctly')

  base: OK (sys=2426 chars, usr=366 chars)
  cot_detailed: OK (sys=1249 chars, usr=428 chars)
  minimal: OK (sys=197 chars, usr=123 chars)
PASS: All SP prompt variants load and format correctly


## Test 3: All CONFIGS create valid SlidingPuzzle instances

In [4]:
from sliding_puzzle.enviroment import SlidingPuzzle, CONFIGS

for name, state in CONFIGS.items():
    puzzle = SlidingPuzzle(initial_state=state)
    sr = puzzle.compute_progress()
    print(f'  {name}: size={puzzle.grid_size}x{puzzle.grid_size}, SR={sr:.1%}, goal={puzzle.goal_state[:4]}...')

print('PASS: All CONFIGS create valid puzzles')

  2x2: size=2x2, SR=25.0%, goal=[0, 1, 2, 3]...
  3x3 (easiest): size=3x3, SR=0.0%, goal=[0, 1, 2, 3]...
  3x3 (hardest): size=3x3, SR=0.0%, goal=[0, 1, 2, 3]...
  4x4 (easiest): size=4x4, SR=0.0%, goal=[0, 1, 2, 3]...
  4x4 (hardest): size=4x4, SR=0.0%, goal=[0, 1, 2, 3]...
PASS: All CONFIGS create valid puzzles


## Test 4: SR (compute_progress) for both games

In [5]:
from sliding_puzzle.enviroment import SlidingPuzzle, CONFIGS
from tower_of_hanoi.enviroment import TowerOfHanoi

# Sliding Puzzle: goal is [0,1,2,...,8]
# State [0,1,2,3,4,5,6,7,8] = goal = solved
p = SlidingPuzzle(initial_state=CONFIGS["2x2"])  # [2,1,3,0], goal=[0,1,2,3]
print(f"SP 2x2 initial: state={p.get_state()}, goal={p.goal_state}, SR={p.compute_progress():.1%}")

# Check a partially correct state
p2 = SlidingPuzzle(initial_state=CONFIGS["3x3 (easiest)"])
sr = p2.compute_progress()
print(f"SP 3x3 easiest: state={p2.get_state()}, SR={sr:.1%}")

# Tower of Hanoi: 0%
h = TowerOfHanoi(num_disks=3)
assert h.compute_progress() == 0.0
print(f"ToH initial: SR=0%")

# Tower of Hanoi: solved
h2 = TowerOfHanoi(num_disks=3)
for m in [(1,0,2), (2,0,1), (1,2,1), (3,0,2), (1,1,0), (2,1,2), (1,0,2)]:
    h2.move_disk(*m)
assert h2.compute_progress() == 1.0
print("ToH solved: SR=100%")

print("PASS: compute_progress works for both games")

SP 2x2 initial: state=[2, 1, 3, 0], goal=[0, 1, 2, 3], SR=25.0%
SP 3x3 easiest: state=[1, 2, 5, 6, 3, 4, 7, 8, 0], SR=0.0%
ToH initial: SR=0%
ToH solved: SR=100%
PASS: compute_progress works for both games


## Test 5: program.md files exist

In [6]:
import os

repo_root = os.path.dirname(os.path.abspath(''))

for f in ['program.md', 'program_fallback.md']:
    path = os.path.join(repo_root, f)
    assert os.path.exists(path), f'{f} not found'
    content = open(path).read()
    assert 'results.tsv' in content, f'{f} missing results.tsv reference'
    assert 'uv run src/main.py' in content, f'{f} missing run command'
    print(f'  {f}: OK ({len(content)} chars)')

print('PASS: program files exist and contain key instructions')

  program.md: OK (4233 chars)
  program_fallback.md: OK (5124 chars)
PASS: program files exist and contain key instructions


## Test 6: run_autoresearch.sh exists and is executable

In [7]:
import os, stat

script = os.path.join(os.path.dirname(os.path.abspath('')), 'scripts', 'run_autoresearch.sh')
assert os.path.exists(script), 'run_autoresearch.sh not found'
mode = os.stat(script).st_mode
assert mode & stat.S_IXUSR, 'run_autoresearch.sh not executable'
content = open(script).read()
assert 'claude -p' in content, 'missing claude -p invocation'
assert 'analyze_autoresearch' in content, 'missing analysis wrap-up'
print(f'  run_autoresearch.sh: OK, executable, has claude + analysis')
print('PASS: sbatch script ready')

  run_autoresearch.sh: OK, executable, has claude + analysis
PASS: sbatch script ready


---
# Cluster Tests (7-9)
Run on GPU node. Auto-skip if no GPU available.

```bash
srun --partition=dev_gpu_h100 --time=00:10:00 --gres=gpu:1 --mem=64G \
  uv run jupyter nbconvert --to notebook --execute --inplace notebooks/test_autoresearch.ipynb
```

## Test 7: GPU visible

In [8]:
import subprocess, shutil

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    print(result.stdout.strip())
    assert result.returncode == 0, 'nvidia-smi failed'
    print('PASS: GPU visible')
else:
    print('SKIP: no nvidia-smi (not on GPU node)')

SKIP: no nvidia-smi (not on GPU node)


## Test 8: Claude Code works on this node

In [9]:
import subprocess, shutil

if shutil.which('claude'):
    # Version check
    result = subprocess.run(['claude', '--version'], capture_output=True, text=True)
    print(f'claude version: {result.stdout.strip()}')
    assert result.returncode == 0, f'claude --version failed: {result.stderr}'

    # Headless response check
    result = subprocess.run(
        ['claude', '-p', 'respond with exactly: HELLO', '--dangerously-skip-permissions', '--model', 'sonnet'],
        capture_output=True, text=True, timeout=60,
    )
    print(f'claude response: {result.stdout.strip()[:100]}')
    assert result.returncode == 0, f'claude -p failed: {result.stderr}'
    assert 'HELLO' in result.stdout.upper(), f'unexpected: {result.stdout[:200]}'
    print('PASS: Claude Code works → use Mode A (sbatch)')
else:
    print('SKIP: claude not installed (use Mode B: tmux + srun)')

claude version: 2.1.76 (Claude Code)


claude response: HELLO
PASS: Claude Code works → use Mode A (sbatch)


## Test 9: main.py runs on GPU (quick 5-step test)

In [10]:
import subprocess, os, shutil

if shutil.which('nvidia-smi'):
    repo_root = os.path.dirname(os.path.abspath(''))
    result = subprocess.run(
        ['uv', 'run', 'src/main.py', '--game', 'tower_of_hanoi', '--num_disks', '3', '--max_steps', '5'],
        capture_output=True, text=True, timeout=300, cwd=repo_root,
    )
    print('stdout (last 5 lines):')
    for line in result.stdout.strip().split('\n')[-5:]:
        print(f'  {line}')
    if result.returncode != 0:
        print(f'stderr (last 300 chars): {result.stderr[-300:]}')
    assert 'Success Rate' in result.stdout, 'No Success Rate in output'
    print('PASS: main.py runs on GPU')
else:
    print('SKIP: no GPU (run on cluster to test main.py)')

SKIP: no GPU (run on cluster to test main.py)


## Summary

**Local tests (no GPU):**
- Test 1: ToH YAML prompt variants
- Test 2: SP YAML prompt variants
- Test 3: All CONFIGS valid
- Test 4: compute_progress() both games
- Test 5: program.md + program_fallback.md exist
- Test 6: run_autoresearch.sh exists + executable

**Cluster tests (GPU):**
- Test 7: GPU visible
- Test 8: Claude Code works → determines Mode A vs B
- Test 9: main.py runs on GPU

All local pass + test 8 pass → **Mode A** (sbatch)
All local pass + test 8 skip → **Mode B** (tmux + srun)